In [ ]:
import pandas as pd

df = pd.read_csv('/content/sample_data/100_Unique_QA_Dataset.csv')

In [ ]:
df.head(), df.shape

(                                          question      answer
 0                   What is the capital of France?       Paris
 1                  What is the capital of Germany?      Berlin
 2               Who wrote 'To Kill a Mockingbird'?  Harper-Lee
 3  What is the largest planet in our solar system?     Jupiter
 4   What is the boiling point of water in Celsius?         100,
 (90, 2))

In [ ]:
# tokenize data into vector
import string

def tokenize(text):
  text = text.lower()  # lowercase
  text = text.translate(str.maketrans('', '', string.punctuation))
  return text.split()

# check
tokenize(df.iloc[0][0])

/tmp/ipykernel_978/651586236.py:10: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  tokenize(df.iloc[0][0])


['what', 'is', 'the', 'capital', 'of', 'france']

In [ ]:
# build vocabulary
vocab = {'<UNK>': 0}

def build_vocab(row):
  query, answer = row['question'], row['answer']

  tokenized_query = tokenize(query)
  tokenized_ans = tokenize(answer)

  merged_token = tokenized_query + tokenized_ans
  for word in merged_token:
    if word not in vocab:
      vocab[word] = len(vocab)

In [ ]:
df.apply(build_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [ ]:
vocab, len(vocab)

({'<UNK>': 0,
  'what': 1,
  'is': 2,
  'the': 3,
  'capital': 4,
  'of': 5,
  'france': 6,
  'paris': 7,
  'germany': 8,
  'berlin': 9,
  'who': 10,
  'wrote': 11,
  'to': 12,
  'kill': 13,
  'a': 14,
  'mockingbird': 15,
  'harperlee': 16,
  'largest': 17,
  'planet': 18,
  'in': 19,
  'our': 20,
  'solar': 21,
  'system': 22,
  'jupiter': 23,
  'boiling': 24,
  'point': 25,
  'water': 26,
  'celsius': 27,
  '100': 28,
  'painted': 29,
  'mona': 30,
  'lisa': 31,
  'leonardodavinci': 32,
  'square': 33,
  'root': 34,
  '64': 35,
  '8': 36,
  'chemical': 37,
  'symbol': 38,
  'for': 39,
  'gold': 40,
  'au': 41,
  'which': 42,
  'year': 43,
  'did': 44,
  'world': 45,
  'war': 46,
  'ii': 47,
  'end': 48,
  '1945': 49,
  'longest': 50,
  'river': 51,
  'nile': 52,
  'japan': 53,
  'tokyo': 54,
  'developed': 55,
  'theory': 56,
  'relativity': 57,
  'alberteinstein': 58,
  'freezing': 59,
  'fahrenheit': 60,
  '32': 61,
  'known': 62,
  'as': 63,
  'red': 64,
  'mars': 65,
  'author':

In [ ]:
# build sequence of indices
def seq_indices(text, vocab):
  sequences = []
  for word in tokenize(text):
    if word in vocab:
      sequences.append(vocab[word])
    else:
      sequences.append(vocab['<UNK>'])
  return sequences

In [ ]:
seq_indices('What is the capital of argentina?', vocab)   # working fine

[1, 2, 3, 4, 5, 0]

In [ ]:
# build data class
import torch
from torch.utils.data import Dataset, DataLoader

class QnADataset:
  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, idx):
    query = self.df.iloc[idx]['question']
    answer = self.df.iloc[idx]['answer']
    seq_query = seq_indices(query, self.vocab)
    seq_answer = seq_indices(answer, self.vocab)
    return torch.tensor(seq_query), torch.tensor(seq_answer)

In [ ]:
dataset = QnADataset(df, vocab)
dataset[0]

(tensor([1, 2, 3, 4, 5, 6]), tensor([7]))

In [ ]:
train_loader = DataLoader(dataset, batch_size=1, shuffle=True)

In [ ]:
for query, ans in train_loader:
  print(query, ans)

print(len(train_loader))

tensor([[  1,   2,   3, 180, 181, 182, 183]]) tensor([[184]])
tensor([[ 1,  2,  3,  4,  5, 73]]) tensor([[74]])
tensor([[  1,   2,   3,  17, 115,  83,  84]]) tensor([[116]])
tensor([[ 42,   2,   3, 274, 211, 275]]) tensor([[276]])
tensor([[42, 86, 87, 88, 89, 39, 90]]) tensor([[91]])
tensor([[ 10, 140,   3, 141, 270,  93, 271,   5,   3, 272]]) tensor([[273]])
tensor([[ 42, 290, 291, 118, 292, 158, 293, 294]]) tensor([[295]])
tensor([[ 10,  11, 157, 158, 159]]) tensor([[160]])
tensor([[ 78,  79, 195,  81,  19,   3, 196, 197, 198]]) tensor([[199]])
tensor([[  1,   2,   3,   4,   5, 113]]) tensor([[114]])
tensor([[  1,   2,   3, 212,   5,  14, 213, 214]]) tensor([[215]])
tensor([[10, 75, 76]]) tensor([[77]])
tensor([[ 1,  2,  3,  4,  5, 53]]) tensor([[54]])
tensor([[ 1,  2,  3, 50, 51, 19,  3, 45]]) tensor([[52]])
tensor([[ 10,  11, 189, 158, 190]]) tensor([[191]])
tensor([[ 42, 167,   2,   3,  17, 168, 169]]) tensor([[170]])
tensor([[ 10, 140,   3, 141, 142,  12, 143,  83,   3, 144]]) te

## RNN architecture

In [ ]:
from torch import nn

class SimpleRNN(nn.Module):
  def __init__(self, vocab_size, input=50):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim=input)
    self.model = nn.RNN(input, 64, batch_first=True)
    self.fully_conn = nn.Linear(64, vocab_size)

  def forward(self, text):
    embedded = self.embedding(text)
    _, final = self.model(embedded)
    output = self.fully_conn(final.squeeze(0))
    return output


In [ ]:
# set hyperparameters
lr = 0.001
epochs = 20

model = SimpleRNN(len(vocab))

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

## Training loop

In [ ]:
# debugging time;)
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
print("shape of a:", a.shape)
b = x(a)
print("shape of b:", b.shape)
c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e = z(d.squeeze(0))

print("shape of e:", e.shape)

shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [ ]:
for e in range(epochs):
  total_loss = 0
  for query, answer in train_loader:
    optimizer.zero_grad()

    # forward pass
    z = model(query)

    # loss calculation
    loss = criterion(z, answer[0])

    # gradients calculation
    loss.backward()

    # weight updates
    optimizer.step()

    # total loss
    total_loss += loss

  print(f"Epochs: {e + 1} - Loss: {total_loss:4f}")


Epochs: 1 - Loss: 523.422974
Epochs: 2 - Loss: 455.155914
Epochs: 3 - Loss: 376.154602
Epochs: 4 - Loss: 311.004395
Epochs: 5 - Loss: 258.229614
Epochs: 6 - Loss: 209.579422
Epochs: 7 - Loss: 166.112656
Epochs: 8 - Loss: 129.422333
Epochs: 9 - Loss: 99.859352
Epochs: 10 - Loss: 77.096649
Epochs: 11 - Loss: 59.728733
Epochs: 12 - Loss: 46.912338
Epochs: 13 - Loss: 38.327301
Epochs: 14 - Loss: 31.467840
Epochs: 15 - Loss: 25.964430
Epochs: 16 - Loss: 21.828785
Epochs: 17 - Loss: 18.634895
Epochs: 18 - Loss: 15.956848
Epochs: 19 - Loss: 13.849028
Epochs: 20 - Loss: 11.950788


In [ ]:
def predict(model, query, threshold=0.5):
  num_query = seq_indices(query, vocab)
  tensor_query = torch.tensor(num_query).reshape(1, len(num_query))
  output = model(tensor_query)

  # convert output into probabilty
  prob = nn.functional.softmax(output, dim=1)
  value, idx = torch.max(prob, dim=1)
  if value < threshold: return "Jaa puchkee aa..."

  return list(vocab.keys())[idx]

In [ ]:
predict(model, "what is the largest planet in our solar system?")

'jupiter'